<div dir="rtl" align="right">

# استخراجُ سماتِ قُوّةِ النطاقِ

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نَحسبُ سماتِ قُوّةِ نطاقَيْ ألفا و بيتا لِكلِّ قناةٍ بِاستخدامِ طريقةِ Welch، مُنتجينَ 44 سمةً لِكلِّ تجربةٍ.

## ماذا يَعمَلُ هذا الدفترُ؟

يَحسبُ كثافةَ الطيفِ الطاقيَّ لِكلِّ قناةٍ وتجربةٍ، ويُكاملُ على نطاقَيْ ألفا (8-13 Hz) و بيتا (13-30 Hz).

## المُخرجاتُ المُتوقّعةُ

- خريطةٌ حراريّةٌ لِمصفوفةِ السماتِ تُظهرُ قيمَ القُوّةِ عبرَ التجاربِ والسماتِ
- مخططٌ شريطيٌّ لِمتوسطِ القُوّةِ لِكلِّ قناةٍ (ألفا + بيتا مُجتمعةً)
- 44 سمةً إجمالاً (22 قناةً × 2 نطاقاً)

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 2 |
| FS | 250 |
| bands | alpha (8-13), beta (13-30) |

</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'After filtering - X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. حسابُ سماتِ قُوّةِ النطاقِ

</div>


In [ ]:
from scipy.signal import welch

FS = 250
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

print(f'Feature matrix shape: {features.shape}')

mean_power = np.zeros(n_channels)
for ch in range(n_channels):
    alpha_idx = ch * len(BANDS) + 0
    beta_idx = ch * len(BANDS) + 1
    mean_power[ch] = np.mean(features[:, alpha_idx]) + np.mean(features[:, beta_idx])

print(f'Mean power per channel computed for {n_channels} channels')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- الخريطةُ الحراريّةُ تُظهرُ تَغيّراً في القُوّةِ عبرَ التجاربِ والسماتِ
- بعضُ القنواتِ لها قُوّةٌ مُتوسّطةٌ أعلى من غيرِها
- نطاقا ألفا و بيتا يَلتقطانِ النشاطَ المُتعلّقَ بِالتخيّلِ الحركيّ

</div>


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Feature Matrix (first 50 trials x 44 features)',
    'Mean Power per Channel'))

fig.add_trace(go.Heatmap(z=features[:50, :], colorscale='Viridis', name='Features', showscale=True), row=1, col=1)
fig.add_trace(go.Bar(x=list(range(n_channels)), y=mean_power, marker_color='steelblue', name='Mean Power'), row=2, col=1)

fig.update_xaxes(title_text='Feature (channel x band)', row=1, col=1)
fig.update_yaxes(title_text='Trial', row=1, col=1)
fig.update_xaxes(title_text='Channel', row=2, col=1)
fig.update_yaxes(title_text='Mean Power (alpha + beta)', row=2, col=1)
fig.update_layout(height=800, showlegend=False, title_text='Band Power Features for ML')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- سماتُ قُوّةِ النطاقِ تمثيلٌ بسيطٌ لكنّهُ فعّالٌ لِتصنيفِ EEG
- 44 سمةً (22 قناةً × 2 نطاقاً) تَلتقطُ المعلوماتَ الطيفيّةَ في نطاقاتِ التخيّلِ الحركيّ
- طريقةُ Welch تُوفّرُ تقديراً مُتيناً لِكثافةِ الطيفِ الطاقيِّ لِاستخراجِ السماتِ

</div>
